# 0.1 LangChain Essentials

**AIME-Con 2026 · Generative AI for Item Development**

**What We'll do**
Meet the eight core components of LangChain that the rest of the day is built from, one short
section each, in the same order as the *Core LangChain Components* slide:

**Models → Messages → Tools → Agents → Structured output → Short-term memory → Streaming → Middleware**



## What LangChain is, and what it is not

**It is** a thin standard layer over the model providers, plus the building blocks every LLM
application ends up needing: messages, tools, structured output, memory, and agents. In LangChain
1.x the center of gravity is `create_agent` in the docs' own words, *Agent = Model + Harness*,
where the harness is the prompt, the tools, and the middleware around the model loop.

**It is not** a model, and it is not required. Every call in this workshop could be made with the
provider's own SDK. We use LangChain for three practical reasons:

1. **Provider independence.** Model IDs get retired and free tiers change. Swapping Gemini for Groq
   is one line, so the notebooks outlive both.
2. **The pieces already fit together.** A retriever feeds a prompt, the prompt feeds a model, and
   the model returns a typed object without glue code you have to maintain.
3. **Governance lives in the agent layer**. Call limits, PII redaction, and human
   approval gates ship as middleware.

And one thing it never is: a validity argument. A framework organizes the calls; it does not
evaluate what comes back.


## ⚙️ Setup

In [3]:
# Colab restart is not needed for these packages.
%pip install -q -U langchain langchain-google-genai langchain-groq

import langchain
import langchain_core

print(f"langchain      {langchain.__version__}")
print(f"langchain-core {langchain_core.__version__}")

if not langchain.__version__.startswith("1."):
    print("\nSTOP: this workshop uses the LangChain 1.x API. Re-run this cell; if the version")
    print("is still not 1.x, restart the runtime (Runtime > Restart session) and run it again.")

langchain      1.4.2
langchain-core 1.6.5


### Choosing a model that still exists

Model IDs change fast. Google has announced that the Gemini 2.5 family shuts down in October
2026, and Groq retires models every few months. Instead of hard-coding one Gemini ID, the cell
below asks **your key** which models it can use today and picks the first one from a preference
list. If none of the preferred IDs is available, it falls back to the newest general Flash model
your key can see.

If a later call fails with a `429` and a quota limit of 0, that model is not on your free tier:
put a different ID at the front of `PREFERRED_GEMINI` and re-run this cell.

In [4]:
import os

from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Preference order. Edit this list, not the rest of the notebook.
PREFERRED_GEMINI = ["gemini-3.6-flash", "gemini-3.5-flash", "gemini-2.5-flash"]
GROQ_MODEL = "openai/gpt-oss-120b"     # Groq's production tier; good at tool calling


def pick_gemini_model(preferred=PREFERRED_GEMINI):
    """Return the first preferred Gemini model this key can call, or the newest Flash model."""
    try:
        from google import genai
        client = genai.Client()
        available = {
            m.name.replace("models/", "")
            for m in client.models.list()
            if not m.supported_actions or "generateContent" in m.supported_actions
        }
        for name in preferred:
            if name in available:
                return name
        skip = ("lite", "preview", "image", "tts", "live", "audio", "exp")
        flash = sorted(n for n in available if "flash" in n and not any(s in n for s in skip))
        if flash:
            return flash[-1]
    except Exception as e:
        print(f"Could not list models ({type(e).__name__}); using the first preference.")
    return preferred[0]


def diagnose(e):
    """Turn an exception into the most likely cause, in plain language."""
    msg = str(e).lower()
    if "429" in msg or "quota" in msg or "rate limit" in msg or "resource_exhausted" in msg:
        return "Rate limit or quota. Wait 60 seconds; if the quota limit is 0, change the model ID."
    if "404" in msg or "not found" in msg or "decommission" in msg:
        return "Model ID not available. Re-run the setup cell or edit PREFERRED_GEMINI / GROQ_MODEL."
    if "401" in msg or "403" in msg or "api key" in msg or "permission" in msg:
        return "Key problem: missing, mistyped, or Notebook access not toggled on in Secrets."
    return "Unexpected error - read the details line."


GEMINI_MODEL = pick_gemini_model()
print("Gemini model:", GEMINI_MODEL)
print("Groq model:  ", GROQ_MODEL)

Gemini model: gemini-3.6-flash
Groq model:   openai/gpt-oss-120b


## 1️. Models

The one object everything else attaches to. `init_chat_model` builds a chat model from a
`"provider:model"` string, so the provider is an argument rather than an import.

In [5]:
from langchain.chat_models import init_chat_model

# GEMINI_MODEL came from the setup cell, so this follows your key rather than a
# hard-coded ID. The "provider:model" string is the short form of
# init_chat_model(GEMINI_MODEL, model_provider="google_genai").
model = init_chat_model(f"google_genai:{GEMINI_MODEL}")

response = model.invoke("In two sentences, why do item writers avoid 'all of the above'?")
print(response.text)

Item writers avoid "all of the above" because it allows test-takers to identify the correct answer using only partial knowledge, such as recognizing that at least two choices are true. Additionally, this option prevents assessment platforms from randomizing answer choices and reduces the overall validity of the question.


In [6]:
# Anything the provider accepts can be passed straight through.
model = init_chat_model(
    f"google_genai:{GEMINI_MODEL}",
    temperature=0.7,     # see the note below: some Gemini 3.x models ignore this
    timeout=30,          # seconds before the call gives up
    max_tokens=1000,     # ceiling on the response
    max_retries=6,       # default; raise it on an unreliable connection
)

print(f"configured: {GEMINI_MODEL}")

configured: gemini-3.6-flash


In [7]:
from langchain.chat_models import init_chat_model

gemini = init_chat_model(f"google_genai:{GEMINI_MODEL}", temperature=0.4)
groq = init_chat_model(f"groq:{GROQ_MODEL}", temperature=0.4)

question = "In one sentence, what is a distractor in a multiple-choice item?"

for name, model in [("gemini", gemini), ("groq  ", groq)]:
    try:
        print(f"{name}: {model.invoke(question).text.strip()[:200]}")
    except Exception as e:
        print(f"{name}: call failed. {diagnose(e)}\n        Details: {type(e).__name__}: {e}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


gemini: In a multiple-choice item, a distractor is an incorrect response option designed to plausibly lure test-takers who do not know the correct answer.
groq  : A distractor is an incorrect answer choice in a multiple‑choice item that is crafted to appear plausible and divert test‑takers away from the correct answer.


Two providers, two free tiers, one interface. The only difference between those objects is a
string.

**Use `.text`, not `.content`.** Newer reasoning models (Gemini 3.x, gpt-oss) can return their
answer as a *list of content blocks* — reasoning, text, tool calls — rather than a plain string.
`.content` then gives you that raw list; `.text` always gives you just the text. Every notebook in
this workshop reads `.text`.

**If you see `uses fixed sampling defaults; temperature will be ignored`** — that warning is
expected and harmless. Some Gemini 3.x models fix their own sampling parameters, so `temperature`
is accepted and then dropped. It matters for one thing only: you cannot make those models more
deterministic by setting `temperature=0`. Notebook 3.0 depends on repeatable judging, so check
this warning before you pick the model you will judge items with.

You will also see the provider classes used directly in later notebooks —
`ChatGoogleGenerativeAI(model=...)`, `ChatGroq(model=...)`. They are equivalent: the direct class
exposes provider-specific arguments, `init_chat_model` gives you the one-string swap.

## 2️. Messages

A chat model takes a list of messages with roles. A bare string works because LangChain wraps it
in a `HumanMessage` for you but once you want to separate standing instructions from the
request, you build the list yourself.

In [8]:
from langchain.messages import AIMessage, HumanMessage, SystemMessage

messages = [
    SystemMessage("You are an experienced 8th grade science item writer. Be concise."),
    HumanMessage("Name one common student misconception about photosynthesis."),
]

try:
    response = gemini.invoke(messages)
    print("type:   ", type(response).__name__)
    print("text:   ", response.text.strip())

    # Token accounting comes back on the response useful when you start
    # measuring what your context actually costs (notebook 1.1).
    if getattr(response, "usage_metadata", None):
        u = response.usage_metadata
        print(f"tokens:  input {u.get('input_tokens')}, output {u.get('output_tokens')}")
except Exception as e:
    print(f"Call failed. {diagnose(e)}\n  Details: {type(e).__name__}: {e}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


type:    AIMessage
text:    Students often believe that **plants absorb their food (or gain their mass) from the soil through their roots**, rather than producing it themselves from carbon dioxide and water using light energy.
tokens:  input 24, output 455


| Role | What it carries |
|---|---|
| `SystemMessage` | Standing instructions: who the model is, what rules always apply |
| `HumanMessage` | The request for this turn |
| `AIMessage` | What the model said, including any tool calls it wants to make |
| `ToolMessage` | The result of a tool the model asked for (you meet it in section 4) |

The **system message is where an item-writing program's standing policy lives**: the register, the
option conventions, the things that must never appear in a stem. Notebook 1.1 builds it
deliberately.

You will also see messages written as plain dictionaries,
`{"role": "user", "content": "..."}` agents accept both forms.

## 3️. Tools

A tool is a Python function the model can **ask** to run. The model never runs your code. It reads
a description of the tool name, docstring, arguments and replies with a request; your code (or
the agent, in section 4) runs the function.

`@tool` builds that description from the function itself. Print it: that is everything the model
knows about your tool.

In [12]:
from langchain.tools import tool

STANDARDS = {
    "MS-LS1-6": "Construct a scientific explanation based on evidence for the role of "
                "photosynthesis in the cycling of matter and flow of energy into and out of organisms.",
    "8.EE.C.7": "Solve linear equations in one variable.",
}


@tool
def lookup_standard(code: str) -> str:
    """Return the official text of a Grade 8 standard, given its code (e.g. 'MS-LS1-6')."""
    return STANDARDS.get(code.strip().upper(), f"No standard found with code {code}.")


print("name:       ", lookup_standard.name)
print("description:", lookup_standard.description)
print("arguments:  ", lookup_standard.args)

# A tool is also just a function you can call yourself:
print("\ndirect call:", lookup_standard.invoke({"code": "8.EE.C.7"}))

name:        lookup_standard
description: Return the official text of a Grade 8 standard, given its code (e.g. 'MS-LS1-6').
arguments:   {'code': {'title': 'Code', 'type': 'string'}}

direct call: Solve linear equations in one variable.


Now hand the tool's description to a model with `bind_tools` and ask a question that needs it.
Watch what comes back: not an answer, but a **request** to call the tool.

In [10]:
model_with_tools = groq.bind_tools([lookup_standard])

try:
    ai_msg = model_with_tools.invoke("What does standard MS-LS1-6 ask students to do?")
    print("text:      ", repr(ai_msg.text[:120]))
    print("tool_calls:", ai_msg.tool_calls)
except Exception as e:
    print(f"Call failed. {diagnose(e)}\n  Details: {type(e).__name__}: {e}")

text:       ''
tool_calls: [{'name': 'lookup_standard', 'args': {'code': 'MS-LS1-6'}, 'id': 'fc_aff9e5e2-a0e7-43b6-8ec4-95984fac3fa4', 'type': 'tool_call'}]


The model asked for `lookup_standard` with `code="MS-LS1-6"` and stopped. Nothing ran. Something
has to execute the call, send the result back as a `ToolMessage`, and ask the model again and
keep doing that until the model answers without asking for a tool. That loop is an **agent**.

## 4️. Agents

`create_agent` is that loop: call the model, run the tools it asks for, repeat, stop when it
answers. Five lines.

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    groq,
    tools=[lookup_standard],
    system_prompt="You write assessment items. Look up a standard before writing for it.",
)


def show_trace(result, width=110):
    """Print every message in a finished run: who spoke, which tool, what came back."""
    for m in result["messages"]:
        kind = type(m).__name__.replace("Message", "")
        if getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"  {kind:<7}-> asks for {tc['name']}({tc['args']})")
        elif m.text.strip():
            print(f"  {kind:<7}   {' '.join(m.text.split())[:width]}")


try:
    out = agent.invoke({"messages": [{"role": "user",
                                      "content": "What does standard MS-LS1-6 ask students to do?"}]})
    show_trace(out)
except Exception as e:
    print(f"Agent call failed. {diagnose(e)}\n  Details: {type(e).__name__}: {e}")

  Human     What does standard MS-LS1-6 ask students to do?
  AI     -> asks for lookup_standard({'code': 'MS-LS1-6'})
  Tool      Construct a scientific explanation based on evidence for the role of photosynthesis in the cycling of matter a
  AI        **MS‑LS1‑6** (Middle School Life Science, Standard 6) asks students to: > **“Construct a scientific explanatio


Read the trace from top to bottom: the request, the model asking for the tool, the tool's answer
(a `ToolMessage`), and the model's final reply built from it. The agent decided on its own to look
the standard up rather than paraphrase it from memory which is exactly what the system prompt
asked for.

Under the hood `create_agent` compiles and runs a LangGraph graph for you, so you get the loop
without writing the nodes and edges yourself. For a genuinely custom topology you can still drop
down to `StateGraph` directly.

## 5️. Structured output

The piece that matters most for item development. Declare the shape you want as a Pydantic model,
bind it to the chat model, and what comes back is an object with typed fields rather than a
paragraph you have to parse.

Take the shape from an item format you already use. The schema below is the anatomy of a
board-style single-best-answer item clinical vignette, focused stem, lettered options, key,
content category, critique the format a standard sample item is written in. If
your program publishes an item format, that document *is* your schema; you are writing down the
structure your item writers already follow.

In [13]:
from typing import List, Literal

from pydantic import BaseModel, Field

Label = Literal["A", "B", "C", "D", "E"]


class Option(BaseModel):
    label: Label
    text: str = Field(description="Option text, no leading label")
    is_key: bool


class BoardStyleItem(BaseModel):
    """A single-best-answer item in the shape of a published board-style sample."""
    vignette: str = Field(description="Clinical scenario: age, sex, site of care, presenting "
                                      "symptoms, duration, vital signs, findings")
    stem: str = Field(description="The question asked of the examinee")
    options: List[Option] = Field(min_length=5, max_length=5)
    key: Label
    content_category: str = Field(description="e.g. Cardiovascular System")
    critique: str = Field(description="Why the key is best and why each distractor is not")


structured = gemini.with_structured_output(BoardStyleItem)

try:
    item = structured.invoke(
        "Write one single-best-answer item for a physician assistant certification exam "
        "on the initial outpatient management of community-acquired pneumonia in an adult."
    )

    # Printed in the layout of the published sample it was modeled on.
    print(item.vignette, "\n")
    print(item.stem)
    for o in item.options:
        print(f"{o.label}. {o.text}")
    print(f"\nAnswer: {item.key}")
    print(f"Medical Content Category: {item.content_category}")
    print(f"\nCritique: {' '.join(item.critique.split())[:280]} ...")
except Exception as e:
    print(f"Call failed. {diagnose(e)}\n  Details: {type(e).__name__}: {e}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


A 34-year-old man presents to an urgent care clinic with a 3-day history of fever, chills, a productive cough with rust-colored sputum, and right-sided pleuritic chest pain. He has no significant past medical history, takes no medications, has no known drug allergies, and has not received antibiotics in the past 90 days. Vital signs reveal a temperature of 38.3°C (101.0°F), blood pressure of 124/76 mm Hg, heart rate of 84/min, respiratory rate of 18/min, and oxygen saturation of 97% on room air. Physical examination demonstrates bronchial breath sounds and crackles over the right lung base. Chest radiography confirms a right lower lobe consolidation. The patient appears non-toxic and is determined to be suitable for outpatient management. 

Which of the following is the most appropriate initial oral antimicrobial therapy for this patient?
A. Amoxicillin 1,000 mg three times daily
B. Azithromycin 500 mg on day 1, followed by 250 mg once daily
C. Ciprofloxacin 500 mg twice daily
D. Levof

`item.key` is a string you can count. `item.options` is a list you can iterate.
`item.content_category` is a field you can group a blueprint by. Nothing was parsed, and the five
options are a constraint rather than a request — a four-option response fails validation instead of
arriving quietly.

That is the difference from the way most generators do this today: pasting an example JSON object
into the prompt and calling `json.loads()` on whatever comes back. The prompt version asks; this
version requires. And when a long instruction contradicts itself *"four options"* in one
paragraph, *"one correct answer and four distractors"* in another the model simply picks a
reading. A schema cannot contradict itself.

**The same idea works on an agent.** Pass `response_format=` to `create_agent` and the final answer
of a multi-step run comes back validated against the schema, in `result["structured_response"]`.

Notebook 5.0 uses this so an agent's output is the same object every other notebook consumes.

In [14]:
# A deliberately small schema for this demo: the point of the cell is `response_format=`,
# and the agent's tool looks up K-12 standards rather than clinical content.
class MiniItem(BaseModel):
    stem: str
    options: List[Option]
    key: Label


typed_agent = create_agent(
    groq,
    tools=[lookup_standard],
    system_prompt="Look up the standard first, then write one item that measures it. "
                  "Use five options labeled A through E.",
    response_format=MiniItem,
)

try:
    result = typed_agent.invoke({"messages": [{"role": "user",
                                               "content": "Write one item for MS-LS1-6."}]})
    typed = result["structured_response"]
    print(type(typed).__name__, "->", typed.stem[:120])
    print("key:", typed.key)
except Exception as e:
    print(f"Call failed. {diagnose(e)}\n  Details: {type(e).__name__}: {e}")

Call failed. Unexpected error - read the details line.
  Details: BadRequestError: Error code: 400 - {'error': {'message': 'json mode cannot be combined with tool/function calling', 'type': 'invalid_request_error', 'param': 'response_format'}}


Notebook 1.0 shows you what life is like without this, and notebook 1.1 builds the full workshop
schema on top of it stem, options with per-option rationales, standard, cognitive level, target
difficulty, grade.

## 6️. Short-term memory

Each `agent.invoke` above started from nothing. To let an agent remember a conversation, give it a
**checkpointer** (where the conversation is saved) and call it with a **thread id** (which
conversation this is). Same thread id, same memory; new thread id, fresh start.

This matters more than it looks. The human approval gate in notebook 5.0 *pauses* an agent
mid-run and resumes it later and it can only do that because the run is checkpointed under a
thread id. This cell is the foundation for that one.

In [15]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    groq,
    tools=[lookup_standard],
    system_prompt="You help an item-writing team. Be brief.",
    checkpointer=InMemorySaver(),        # in production this is a database
)


def ask(text, thread_id):
    config = {"configurable": {"thread_id": thread_id}}
    result = memory_agent.invoke({"messages": [{"role": "user", "content": text}]}, config=config)
    return result["messages"][-1].text.strip()


try:
    print("[thread A] ", ask("This week our team is writing items for standard 8.EE.C.7.", "team-A"))
    print()
    print("[thread A] ", ask("Which standard did I say we are working on?", "team-A"))
    print()
    print("[thread B] ", ask("Which standard did I say we are working on?", "team-B"))
except Exception as e:
    print(f"Call failed. {diagnose(e)}\n  Details: {type(e).__name__}: {e}")

[thread A]  **Standard 8.EE.C.7 – “Solve linear equations in one variable.”**  

- **Core expectation:** Students must be able to isolate the variable and find its value in equations such as \(2x+5=13\), \(\frac{x}{4}=3\), or \(5-2(x-1)=3x+7\).  
- **Typical item formats:**  
  1. **Direct computation** – give an equation and ask for the solution.  
  2. **Word‑problem translation** – present a real‑world scenario, have students write the equation, then solve it.  
  3. **Multiple‑step equations** – include distributive property, combining like terms, or fractions.  
  4. **Checking work** – ask students to verify the solution by substitution.  

**Tips for item writing:**  
- Keep the variable isolated in the answer key (e.g., “\(x = 4\)”).  
- Vary the difficulty by changing the number of steps or the type of coefficients (negative, fractional, or decimal).  
- Ensure the context (if any) is age‑appropriate and mathematically relevant to 8th‑grade life (e.g., budgeting, distance‑rate

Thread A remembers; thread B has never heard of it. `InMemorySaver` keeps everything in this Colab
session and forgets it when the runtime restarts fine for a notebook. A real deployment uses a
database-backed checkpointer, and the same thread id picks the conversation up days later.

## 7️. Streaming

Everything so far waited for the complete answer. Streaming shows output as it is produced —
token by token from a model, or step by step from an agent. It changes nothing about *what* comes
back, only how soon a reader sees it.

In [16]:
# Tokens from a model, as they arrive.
try:
    for chunk in gemini.stream("In two sentences, why do item writers avoid 'all of the above'?"):
        print(chunk.text, end="", flush=True)
    print()
except Exception as e:
    print(f"\nStream failed. {diagnose(e)}\n  Details: {type(e).__name__}: {e}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Item writers avoid "all of the above" because it allows test-takers to deduce the correct answer with only partial knowledge, such as recognizing that two options are true. Additionally, identifying just one incorrect option enables students to immediately eliminate "all of the above," artificially increasing their chances of guessing correctly.


In [17]:
# Steps from an agent, as each one finishes: the model node, then the tools node, then the model again.
try:
    for update in agent.stream(
        {"messages": [{"role": "user", "content": "What does standard 8.EE.C.7 require?"}]},
        stream_mode="updates",
    ):
        for step, data in update.items():
            if not data or "messages" not in data:
                continue
            last = data["messages"][-1]
            if getattr(last, "tool_calls", None):
                print(f"[{step}] asks for {[tc['name'] for tc in last.tool_calls]}")
            else:
                print(f"[{step}] {' '.join(last.text.split())[:110]}")
except Exception as e:
    print(f"Stream failed. {diagnose(e)}\n  Details: {type(e).__name__}: {e}")

[model] asks for ['lookup_standard']
[tools] Solve linear equations in one variable.
[model] asks for ['lookup_standard']
[tools] Solve linear equations in one variable.
[model] **Standard 8.EE.C.7 (Common Core State Standards for Mathematics – Grade 8)** > **Requirement:** *Students mus


**Event streaming (preview).** LangChain v1.3 introduced a newer streaming API with typed
streams — separate iterators for messages, tool calls, and state — instead of the
`stream_mode` dictionaries above. It is still marked *preview* and may change, so the workshop does
not depend on it. For reference, the shape is:

```python
stream = agent.stream_events({"messages": [...]}, version="v3")
for message in stream.messages:
    for token in message.text:
        print(token, end="")
final_state = stream.output
```

Nothing else in the workshop streams; we generate items in batches and read them afterwards. You
need streaming when a person is watching a long response being written — a chat interface, or an
agent working through several steps.

## 8️. Middleware

Middleware hooks run before and after every model call and tool call inside an agent. It is how
LangChain 1.x expresses controls: a policy written as code that runs on every invocation, instead
of a paragraph in a process document.

One example now; notebook 5.0 builds the full set (call limits, PII redaction, human approval,
summarization).

In [18]:
from langchain.agents.middleware import ModelCallLimitMiddleware

budgeted_agent = create_agent(
    groq,
    tools=[lookup_standard],
    system_prompt="Look up every standard you are asked about before answering. Be brief.",
    middleware=[
        # Bounded cost: at most 2 model calls in one run, then stop cleanly.
        ModelCallLimitMiddleware(run_limit=2, exit_behavior="end"),
    ],
)

try:
    out = budgeted_agent.invoke({"messages": [{"role": "user", "content":
        "Look up MS-LS1-6, then look up 8.EE.C.7, then compare them in one sentence."}]})
    # The middleware adds its own stop notice as an AIMessage; don't count it as a model call.
    model_turns = sum(1 for m in out["messages"] if type(m).__name__ == "AIMessage"
                      and not m.text.startswith("Model call limits exceeded"))
    print(f"model calls made: {model_turns}  (limit: 2)\n")
    show_trace(out)
except Exception as e:
    print(f"Call failed. {diagnose(e)}\n  Details: {type(e).__name__}: {e}")

model calls made: 2  (limit: 2)

  Human     Look up MS-LS1-6, then look up 8.EE.C.7, then compare them in one sentence.
  AI     -> asks for lookup_standard({'code': 'MS-LS1-6'})
  Tool      Construct a scientific explanation based on evidence for the role of photosynthesis in the cycling of matter a
  AI     -> asks for lookup_standard({'code': '8.EE.C.7'})
  Tool      Solve linear equations in one variable.
  AI        Model call limits exceeded: run limit (2/2)


The task needs three model calls if the model looks the standards up one at a time (ask for a
lookup, ask for another, then answer). The limit stops the run at two, and the last line of the
trace is the middleware's own stop notice, not the model. If the model asks for both lookups in a
single call, it can finish within the budget either way, it cannot exceed it. "No item-generation run
may exceed N model calls" is now a line of code rather than a hope.

## 🧩 Composition tools: prompt templates and chains

These sit alongside the eight components rather than among them, and notebook 2.0's retrieval
chain is built from both.

Prompt templates and chains are documented in the
API reference instead:

| What you want | Where it is |
|---|---|
| `PromptTemplate`, `ChatPromptTemplate`, `FewShotPromptTemplate`, `MessagesPlaceholder` | [reference.langchain.com/python/langchain-core/prompts](https://reference.langchain.com/python/langchain-core/prompts) |
| Chains: `Runnable`, `RunnableSequence` (what `\|` builds), `RunnableParallel`, `RunnableLambda` | [reference.langchain.com/python/langchain-core/runnables](https://reference.langchain.com/python/langchain-core/runnables) |
| The eight core components | [docs.langchain.com/oss/python/langchain/models](https://docs.langchain.com/oss/python/langchain/models) and its siblings |
| Full page index | [docs.langchain.com/llms.txt](https://docs.langchain.com/llms.txt) |

**LCEL** is the name for the `|` operator. It is documented as part of the Runnable interface
rather than as a topic of its own, so search that reference for `Runnable` rather than for LCEL.
The two reference links above are the authoritative source — documentation on other domains may
describe a different major version of the library.

A **prompt template** is a prompt with holes in it. It stops you pasting f-strings around your
code and ending up with six slightly different versions of your item instruction.

In [19]:
from langchain_core.prompts import ChatPromptTemplate

item_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an experienced {grade} {subject} item writer. Be concise."),
    ("human", "Write one multiple-choice item on {topic}. Four options, mark the key."),
])

# Fill it in without calling the model, to see exactly what would be sent.
filled = item_prompt.invoke({"grade": "8th grade", "subject": "science",
                             "topic": "the role of chlorophyll"})
for m in filled.messages:
    print(f"[{type(m).__name__}] {m.content}")

[SystemMessage] You are an experienced 8th grade science item writer. Be concise.
[HumanMessage] Write one multiple-choice item on the role of chlorophyll. Four options, mark the key.


A **chain** joins pieces with the `|` operator: `prompt | model | parser` means fill the template,
send it, pull the text out. What you get back has `.invoke()`, `.batch()`, and `.stream()` so
going from one item to forty is a method call.

In [20]:
from langchain_core.output_parsers import StrOutputParser

chain = item_prompt | gemini | StrOutputParser()

try:
    print(chain.invoke({"grade": "8th grade", "subject": "science",
                        "topic": "the role of chlorophyll"})[:400])
except Exception as e:
    print(f"Call failed. {diagnose(e)}\n  Details: {type(e).__name__}: {e}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**What is the primary role of chlorophyll in plant leaves?**

A) To absorb sunlight for photosynthesis *
B) To transport water from roots to leaves
C) To store glucose produced by the plant
D) To protect the plant from harmful insects

**Key:** A


In [21]:
# Same chain, three topics. max_concurrency keeps free-tier rate limits happy.
topics = ["the role of chlorophyll", "what plants take in from the air",
          "why leaves change color in autumn"]
inputs = [{"grade": "8th grade", "subject": "science", "topic": t} for t in topics]

try:
    results = chain.batch(inputs, config={"max_concurrency": 2})
    for t, r in zip(topics, results):
        print(f"=== {t} ===\n{r.strip()[:180]}\n")
except Exception as e:
    print(f"Batch failed. {diagnose(e)}\n  Details: {type(e).__name__}: {e}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== the role of chlorophyll ===
**Question:**
What is the primary role of chlorophyll during the process of photosynthesis?

A. To absorb light energy from the Sun  
B. To take in water and minerals from the soil

=== what plants take in from the air ===
**Which gas do plants primarily take in from the air to make their own food during photosynthesis?**

A. Nitrogen  
B. Carbon dioxide  
C. Oxygen  
D. Hydrogen  

**Correct Answer:

=== why leaves change color in autumn ===
**Question:**
During autumn, the leaves on many trees change from green to shades of yellow, orange, and red. Which statement best explains why this color change occurs?

A) Frost 



### Retrieval, in outline (notebook 2.0)

Four objects that snap together. You will build them properly later; for now, just so the imports
look familiar:

```python
from langchain_community.document_loaders import PyPDFLoader         # read source files
from langchain_text_splitters import RecursiveCharacterTextSplitter  # cut into chunks
from langchain_huggingface import HuggingFaceEmbeddings              # chunks -> vectors
from langchain_chroma import Chroma                                  # store and search vectors

retriever = vector_store.as_retriever(search_kwargs={"k": 4})
```

## YOUR TURN

Two minutes. Pick one:

**(a) Swap the model, keep everything else.** Point the chain at Groq and re-run it. Nothing else
changes that is the property you are buying by working through this layer.

**(b) Give the memory agent a longer conversation.** Add a third turn to thread A that depends on
both earlier turns, and check it still answers correctly.

In [ ]:
# YOUR TURN (a) - swap the model, keep everything else.
my_chain = item_prompt | groq | StrOutputParser()

try:
    print(my_chain.invoke({"grade": "8th grade", "subject": "science",
                           "topic": "what plants take in from the air"}).strip()[:400])
except Exception as e:
    print(f"Call failed. {diagnose(e)}\n  Details: {type(e).__name__}: {e}")

# YOUR TURN (b) - uncomment and edit:
# print(ask("Look that standard up and suggest one misconception to target.", "team-A"))

## Where each component shows up today

The same map as the *Where It Shows Up* slide.

| Notebook | What you do | Components used |
|---|---|---|
| **0.1** LangChain Essentials | Tour the core components, one section each | All of them |
| **1.0** Prompt Engineering | Zero-shot, few-shot, chain-of-thought and their ceiling | Models |
| **1.1** Context Engineering | Blueprint, constraints, exemplars into typed items | Messages, Structured output |
| **2.0** RAG | Items grounded in your curriculum, with citations | Retrieval, Prompt templates, Structured output |
| **3.0** Evaluation and QC | Deterministic checks, a calibrated LLM judge, triage | Structured output, Embeddings |
| **4.0** Fine-Tuning (LoRA) | House style on a small open model | None Hugging Face TRL + PEFT |
| **5.0** Agents and Governance | A tool-using agent with executable controls | Agents, Tools, Memory, Middleware |

See the ITC/ATP *Guidelines for Technology-Based Assessment* (v1.1, July 2025) and the ATP
*Generative AI Primer* (2025).

**Next:** `1.0_Prompt_Engineering.ipynb`